In [3]:
import os, subprocess

def load_amber_env():
    """Load AmberTools + Amber environment into the Jupyter kernel."""
    bash_cmd = """
    module purge
    module load StdEnvACCRE/2023
    module load gcc/12.3 openmpi/4.1.5 cuda/12.2
    module load ambertools/25.0
    module load scipy-stack/2025a
    module load amber/24
    env
    """
    env_output = subprocess.check_output(["bash", "-c", bash_cmd], text=True)
    for line in env_output.splitlines():
        if "=" in line:
            k, v = line.split("=", 1)
            os.environ[k] = v
    print("✅ Amber environment loaded into Jupyter kernel.")


In [4]:
load_amber_env()

The following modules were not unloaded:
  (Use "module --force purge" to unload all):

  1) CCconfig        6)  ucx/1.14.1         11) flexiblas/3.3.1
  2) gentoo/2023     7)  libfabric/1.18.0   12) imkl/2023.2.0
  3) gcccore/.12.3   8)  pmix/4.2.4         13) StdEnvACCRE/2023
  4) gcc/12.3        9)  ucc/1.2.0
  5) hwloc/2.9.1     10) openmpi/4.1.5
Lmod Warning:
-------------------------------------------------------------------------------
The following dependent module(s) are not currently loaded: scipy-stack/2023b
(required by: plumed/2.9.0, ambertools/25.0)
-------------------------------------------------------------------------------




The following have been reloaded with a version change:
  1) ipykernel/2023b => ipykernel/2025a
  2) scipy-stack/2023b => scipy-stack/2025a



✅ Amber environment loaded into Jupyter kernel.


Lmod Warning:
-------------------------------------------------------------------------------
The following dependent module(s) are not currently loaded: scipy-stack/2025a
(required by: amber/24)
-------------------------------------------------------------------------------




Inactive Modules:
  1) arrayfire/3.9.0     2) cudnn/8.9.5.29     3) glfw/3.3.8     4) spdlog/1.11.0

Due to MODULEPATH changes, the following have been reloaded:
  1) ambertools/25.0     2) plumed/2.9.0     3) ucc-cuda/1.2.0     4) ucx-cuda/1.14.1

The following have been reloaded with a version change:
  1) cuda/12.2 => cuda/12.6
  2) cudacore/.12.2.2 => cudacore/.12.6.2
  3) ipykernel/2025a => ipykernel/2023b
  4) nccl/2.18.3 => nccl/2.26.2
  5) scipy-stack/2025a => scipy-stack/2023b



## 🧬 **Master Cα Concatenation — cMD + aMD**

- **ALL_SYSTEMS_Ca_combined.nc** — concatenation of the global **cMD → aMD** Cα trajectories

The **6ONI apo `Ca_combined.prmtop`** is used as the reference topology for the combined trajectory.

In [ ]:
%%bash
set -euo pipefail

BASE="/panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_cMD_12_27_2025"
OUTDIR="/panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_aMD_cMD_12_30_2025"

mkdir -p "$OUTDIR"

echo "🧬 Creating global Cα master trajectory:"
echo "    Order: cMD → aMD"
echo

cpptraj <<EOF
parm ${OUTDIR}/6ONI_apo/Ca_combined.prmtop
trajin /panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_cMD_12_27_2025/6ONI_6ONJ_ALL_PCA/ALL/ALL_SYSTEMS_Ca_combined.nc
trajin /panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_aMD_12_27_2025/6ONI_6ONJ_ALL_PCA/ALL/ALL_SYSTEMS_Ca_combined.nc
trajout ${OUTDIR}/ALL_SYSTEMS_Ca_combined.nc netcdf
run
quit
EOF

echo "🎯 ${OUTDIR}/ALL_SYSTEMS_Ca_combined.nc created successfully."


## 🧬 **Global PCA — Combined cMD + aMD Landscape**

This step performs PCA on the combined **cMD + aMD Cα master trajectory**:

- **Input trajectory:** `ALL_SYSTEMS_Ca_combined.nc`
- **Reference topology:** 6ONI apo `Ca_combined.prmtop`
- Structures are aligned using **Cα atoms from residues 1–275**
- A covariance matrix is calculated from the aligned trajectory
- The first **50 principal components** are extracted
- All frames are projected onto these 50 PCs and written to `pca.dat`

The PCA therefore defines a single common conformational space spanning the complete **cMD + aMD dataset**.

In [6]:
%%bash
set -euo pipefail

BASE="/panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_aMD_cMD_12_30_2025"
SLURM_DIR="${BASE}/Slurm_Scripts"
mkdir -p "$SLURM_DIR"
cd "$SLURM_DIR"

cat > PCA_ALL_GROUPS.slurm <<'EOF'
#!/bin/bash
#SBATCH --job-name=PCA_ALL_GROUPS
#SBATCH --mail-user=mithun.nag.karadi.giridhar@vanderbilt.edu
#SBATCH --mail-type=ALL
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=1
#SBATCH --mem=60G
#SBATCH --gres=gpu:nvidia_titan_x:1
#SBATCH --time=5-00:00:00
#SBATCH --partition=batch_gpu
#SBATCH --account=csb_gpu_acc
#SBATCH --output=PCA_ALL_GROUPS_out_%j.log
#SBATCH --error=PCA_ALL_GROUPS_err_%j.log


module --force purge
module load StdEnvACCRE/2023
module load ambertools/25.0
module load amber/24
export OMP_NUM_THREADS=1

BASE="/panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_aMD_cMD_12_30_2025"

# =====================================================
# GROUP 1 — GLOBAL PCA (cMD + aMD master trajectory)
# =====================================================
echo "===== PCA GROUP 1 : GLOBAL cMD + aMD PCA ====="

cd "${BASE}"

cpptraj <<EOT
parm /panfs/accrepfs.vampire/data/kojetin_lab/Mithun/Publish_cMD_12_27_2025/6ONI_apo/Ca_combined.prmtop
trajin ${BASE}/ALL_SYSTEMS_Ca_combined.nc
autoimage anchor :1-275@CA
rms first :1-275@CA
average :1-275@CA crdset avgstruct
createcrd trajectory
run
crdaction trajectory rms ref avgstruct :1-275@CA
crdaction trajectory matrix covar name covmatrix :1-275@CA
runanalysis diagmatrix covmatrix out evecs.dat vecs 50 name AllModes
crdaction trajectory projection modes AllModes beg 1 end 50 :1-275@CA out pca.dat
quit
EOT

done

echo "🎯 ALL PCA GROUPS COMPLETED SUCCESSFULLY"
EOF

sbatch PCA_ALL_GROUPS.slurm


bash: ml: line 1: syntax error: unexpected end of file
bash: error importing function definition for `ml'
bash: find_port: line 1: syntax error: unexpected end of file
bash: error importing function definition for `find_port'
bash: which: line 1: syntax error: unexpected end of file
bash: error importing function definition for `which'
bash: create_passwd: line 1: syntax error: unexpected end of file
bash: error importing function definition for `create_passwd'
bash: random_number: line 1: syntax error: unexpected end of file
bash: error importing function definition for `random_number'
bash: module: line 1: syntax error: unexpected end of file
bash: error importing function definition for `module'
bash: setup_accre_software_stack: line 1: syntax error: unexpected end of file
bash: error importing function definition for `setup_accre_software_stack'
bash: wait_until_port_used: line 1: syntax error: unexpected end of file
bash: error importing function definition for `wait_until_port_us

Submitted batch job 8050670
